# 06b Silver prices
Cleans bronze.prices, removes duplicate prices, and builds a latest-price view per security.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

clean = spark.table("bronze.prices")
for c in ["security_id", "currency", "price_source"]:
    clean = clean.withColumn(c, F.expr(f"NULLIF(NULLIF(UPPER(TRIM({c})), ''), 'NULL')"))

REJECT_RULES = [
    ("INVALID_DATE",     "TRY_CAST(price_date AS DATE) IS NULL"),
    ("INVALID_PRICE",    "TRY_CAST(price AS DECIMAL(18,4)) IS NULL OR TRY_CAST(price AS DECIMAL(18,4)) <= 0"),
    ("INVALID_CURRENCY", "currency IS NULL OR currency NOT IN ('USD', 'GBP', 'EUR')"),
    ("UNKNOWN_SECURITY", "sec_found IS NULL"),
]
case_sql = "CASE " + " ".join(f"WHEN {cond} THEN '{reason}'" for reason, cond in REJECT_RULES) + " END"

known = (spark.table("bronze.securities")
         .select(F.upper(F.trim("security_id")).alias("sec_found")).distinct())

checked = (clean.join(known, clean.security_id == known.sec_found, "left")
                .withColumn("reject_reason", F.expr(case_sql)))
good = checked.filter("reject_reason IS NULL")
bad  = checked.filter("reject_reason IS NOT NULL").drop("sec_found")

In [0]:
typed = good.select(
    "security_id",
    F.col("price_date").cast("date"),
    F.col("price").cast("decimal(18,4)"),
    "currency", "price_source", "_source_file", "_ingested_at",
)

# one price per security and date: prefer BLOOMBERG, then the most recently loaded
w = Window.partitionBy("security_id", "price_date").orderBy(
    F.when(F.col("price_source") == "BLOOMBERG", 1).otherwise(2),
    F.col("_ingested_at").desc(),
)
ranked = typed.withColumn("rn", F.row_number().over(w))
latest = ranked.filter("rn = 1").drop("rn")
older  = ranked.filter("rn > 1").drop("rn").withColumn("reject_reason", F.lit("DUPLICATE_PRICE"))

In [0]:
older_text = older.select([c if c == "_ingested_at" else F.col(c).cast("string").alias(c) for c in older.columns])
quarantine = bad.unionByName(older_text, allowMissingColumns=True)

latest.write.mode("overwrite").saveAsTable("silver.prices")
quarantine.write.mode("overwrite").saveAsTable("silver.prices_quarantine")

In [0]:
%sql
CREATE OR REPLACE VIEW silver.latest_prices AS
SELECT security_id, price_date, price, currency
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY security_id ORDER BY price_date DESC) AS rn
  FROM silver.prices
)
WHERE rn = 1

In [0]:
%sql
SELECT 'silver' AS tbl, COUNT(*) AS n FROM silver.prices
UNION ALL
SELECT 'quarantine', COUNT(*) FROM silver.prices_quarantine

In [0]:
%sql
SELECT security_id, price_date, reject_reason FROM silver.prices_quarantine ORDER BY security_id

In [0]:
%sql
SELECT * FROM silver.latest_prices ORDER BY security_id